# ZFN Default-Behavior Bottleneck Profile

This notebook profiles the direct Python ZFN search path in `src/sirnaforge/zfn/search.py` to support a reasonable shipped default behavior.

Scope:
- chr3-only exhaustive runs against the Ensembl hg38 chromosome 3 FASTA
- full-size hg38 exhaustive runs against `ensembl_human_hg38_primary`
- no environment-variable tuning or container-only runtime knobs
- worker and shard-size sweeps that reflect the actual in-process code path

The analysis reports:
- wall-clock speedup versus an unsharded serial baseline
- incremental speedup from sharding and worker scaling
- phase shares and theoretical payoff if a phase were made free

Annotation is deliberately excluded so the results stay focused on search-path bottlenecks rather than external GTF I/O or region labeling.

In [ ]:
from __future__ import annotations

import importlib
import json
import os
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

import sirnaforge.models.zfn as zfn_models
import sirnaforge.zfn.search as zfn_search

importlib.reload(zfn_models)
importlib.reload(zfn_search)

from sirnaforge.models.zfn import (
    ZFNAlgorithm,
    ZFNDesignParameters,
    ZFNHalfSiteConstraints,
    ZFNShardingConfig,
    ZFNSpacerConstraints,
)
from sirnaforge.zfn.rank import rank_sites
from sirnaforge.zfn.search import ExhaustiveZFNOffTargetSearcher

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = Path("/home/hovland/sirnaforge/sirnaforge/notebooks")
WORKSPACE_ROOT = NOTEBOOK_DIR.parent
RUN_ROOT = NOTEBOOK_DIR / "zfn_experiment_runs"
ARTIFACT_ROOT = RUN_ROOT / "default_behavior_profile"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

CCR5_LEFT_HALF_SITE = "GTCATCCTCATC"
CCR5_RIGHT_HALF_SITE = "AAACTGCAAAAG"
DEFAULT_SPACERS = [5, 6]
DEFAULT_CHUNK_BP = 12_000_000
HG38_PRIMARY_REFERENCE = "ensembl_human_hg38_primary"
HG38_CHR3_FASTA_URL = (
    "https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.3.fa.gz"
)
AVAILABLE_CPUS = max(1, os.cpu_count() or 1)
AVAILABLE_MEMORY_GB = ExhaustiveZFNOffTargetSearcher._available_memory_gb()
REPEATS_CHR3 = 2
REPEATS_HG38 = 1
RUN_FULL_HG38 = False
TOP_N_SITES = 5000

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT}")
print(f"Available CPUs: {AVAILABLE_CPUS}")
print(f"Available memory GiB: {AVAILABLE_MEMORY_GB}")
print(f"Full hg38 enabled: {RUN_FULL_HG38}")
print("Current typed sharding defaults:", ZFNShardingConfig().model_dump())

## Method

Each case is benchmarked in four layers:

1. Unsharded serial baseline.
2. Sharded serial run at the current default chunk size.
3. Worker sweep at the default chunk size.
4. Chunk-size sweep at the best observed worker count for that case.

The profiling wrapper times the same internal phases used by the production searcher: input resolution, FASTA load, shard planning, shard search, dedupe, ranking, and truncation.

Interpretation rule: if a phase already consumes a fraction $f$ of wall time, the maximum end-to-end speedup from making only that phase free is $1 / (1 - f)$.

In [ ]:
@dataclass(frozen=True)
class BenchmarkCase:
    case_id: str
    label: str
    search_space_reference: str | None = None
    search_space_fasta: str | None = None
    repeats: int = 1
    chunk_candidates: tuple[int, ...] = (DEFAULT_CHUNK_BP,)


def build_worker_candidates(max_cpus: int) -> list[int]:
    candidates = {1, max_cpus}
    power = 1
    while power <= max_cpus:
        candidates.add(power)
        power *= 2
    for extra in (3, 6, 12, 24, 32):
        if extra <= max_cpus:
            candidates.add(extra)
    return sorted(candidates)


def make_params(
    case: BenchmarkCase,
    *,
    max_workers: int,
    chunk_size_bp: int,
    sharding_enabled: bool,
) -> ZFNDesignParameters:
    return ZFNDesignParameters(
        search_space_reference=case.search_space_reference,
        search_space_fasta=case.search_space_fasta,
        left_half_site=CCR5_LEFT_HALF_SITE,
        right_half_site=CCR5_RIGHT_HALF_SITE,
        algorithm=ZFNAlgorithm.ZFN_V2,
        top_n_sites=TOP_N_SITES,
        half_site_constraints=ZFNHalfSiteConstraints(
            max_mismatches=2,
            seed_len_from_fokI=6,
            seed_max_mismatches=1,
            window_stride=1,
        ),
        spacer_constraints=ZFNSpacerConstraints(allowed_spacer_lengths=DEFAULT_SPACERS),
        sharding=ZFNShardingConfig(
            enabled=sharding_enabled,
            chunk_size_bp=chunk_size_bp,
            overlap_bp=50,
            chromosomes=[],
            max_workers=max_workers,
        ),
    )


WORKER_CANDIDATES = build_worker_candidates(AVAILABLE_CPUS)
CASES = [
    BenchmarkCase(
        case_id="chr3",
        label="chr3 only",
        search_space_fasta=HG38_CHR3_FASTA_URL,
        repeats=REPEATS_CHR3,
        chunk_candidates=(2_000_000, 4_000_000, 8_000_000, 12_000_000),
    ),
]
if RUN_FULL_HG38:
    CASES.append(
        BenchmarkCase(
            case_id="hg38_full",
            label="full hg38 primary assembly",
            search_space_reference=HG38_PRIMARY_REFERENCE,
            repeats=REPEATS_HG38,
            chunk_candidates=(8_000_000, 12_000_000, 16_000_000, 24_000_000),
        )
    )

display(pd.DataFrame([asdict(case) for case in CASES]))
print("Worker candidates:", WORKER_CANDIDATES)

In [ ]:
class InstrumentedSearcher(ExhaustiveZFNOffTargetSearcher):
    def profile(self, params: ZFNDesignParameters) -> dict[str, Any]:
        phase_timings: dict[str, float] = {}
        started_at = time.perf_counter()

        phase_start = time.perf_counter()
        fasta_path = self._resolve_search_space_fasta(params)
        phase_timings["resolve_inputs_s"] = time.perf_counter() - phase_start

        phase_start = time.perf_counter()
        chrom_sequences = self._load_fasta(fasta_path)
        phase_timings["load_fasta_s"] = time.perf_counter() - phase_start

        phase_start = time.perf_counter()
        shard_specs = self._build_shard_specs(chrom_sequences, params)
        phase_timings["build_shards_s"] = time.perf_counter() - phase_start

        effective_workers = (
            min(self._recommended_worker_cap(chrom_sequences, params), len(shard_specs)) if shard_specs else 1
        )

        phase_start = time.perf_counter()
        raw_sites = self._run_shard_searches(
            shard_specs=shard_specs,
            chrom_sequences=chrom_sequences,
            params=params,
            workers=effective_workers,
            started_at=phase_start,
        )
        phase_timings["search_shards_s"] = time.perf_counter() - phase_start

        phase_start = time.perf_counter()
        deduped = self._dedupe_sites(raw_sites)
        phase_timings["dedupe_s"] = time.perf_counter() - phase_start

        phase_start = time.perf_counter()
        ranked = rank_sites(deduped, params)
        phase_timings["rank_s"] = time.perf_counter() - phase_start

        phase_start = time.perf_counter()
        truncated = self._truncate_sites(ranked, params.top_n_sites)
        phase_timings["truncate_s"] = time.perf_counter() - phase_start

        total_s = time.perf_counter() - started_at
        return {
            "phase_timings": phase_timings,
            "total_s": total_s,
            "resolved_fasta": str(fasta_path),
            "total_bp": sum(len(seq) for seq in chrom_sequences.values()),
            "requested_workers": params.sharding.max_workers,
            "effective_workers": effective_workers,
            "shard_count": len(shard_specs),
            "raw_sites": len(raw_sites),
            "deduped_sites": len(deduped),
            "ranked_sites": len(ranked),
            "retained_sites": len(truncated),
        }


def run_profile(
    case: BenchmarkCase,
    *,
    profile_name: str,
    sharding_enabled: bool,
    max_workers: int,
    chunk_size_bp: int,
    repeats: int,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for repeat_idx in range(1, repeats + 1):
        print(
            f"[{case.case_id}] {profile_name} repeat {repeat_idx}/{repeats}: "
            f"sharding={sharding_enabled} workers={max_workers} chunk={chunk_size_bp:,}"
        )
        params = make_params(
            case,
            max_workers=max_workers,
            chunk_size_bp=chunk_size_bp,
            sharding_enabled=sharding_enabled,
        )
        result = InstrumentedSearcher().profile(params)
        row = {
            "case_id": case.case_id,
            "case_label": case.label,
            "profile_name": profile_name,
            "repeat": repeat_idx,
            "sharding_enabled": sharding_enabled,
            "requested_workers": max_workers,
            "effective_workers": result["effective_workers"],
            "chunk_size_bp": chunk_size_bp,
            "wall_s": result["total_s"],
            "shard_count": result["shard_count"],
            "total_bp": result["total_bp"],
            "raw_sites": result["raw_sites"],
            "deduped_sites": result["deduped_sites"],
            "ranked_sites": result["ranked_sites"],
            "retained_sites": result["retained_sites"],
            "resolved_fasta": result["resolved_fasta"],
        }
        row.update(result["phase_timings"])
        rows.append(row)
    return rows

In [ ]:
all_rows: list[dict[str, Any]] = []
best_worker_by_case: dict[str, int] = {}

for case in CASES:
    all_rows.extend(
        run_profile(
            case,
            profile_name="baseline_unsharded",
            sharding_enabled=False,
            max_workers=1,
            chunk_size_bp=DEFAULT_CHUNK_BP,
            repeats=case.repeats,
        )
    )
    all_rows.extend(
        run_profile(
            case,
            profile_name="sharded_serial",
            sharding_enabled=True,
            max_workers=1,
            chunk_size_bp=DEFAULT_CHUNK_BP,
            repeats=case.repeats,
        )
    )

    worker_rows_before = len(all_rows)
    for workers in WORKER_CANDIDATES:
        all_rows.extend(
            run_profile(
                case,
                profile_name=f"worker_sweep_w{workers}",
                sharding_enabled=True,
                max_workers=workers,
                chunk_size_bp=DEFAULT_CHUNK_BP,
                repeats=case.repeats,
            )
        )

    worker_frame = pd.DataFrame(all_rows[worker_rows_before:])
    worker_summary = (
        worker_frame.groupby(["requested_workers", "effective_workers"], as_index=False)
        .agg(mean_wall_s=("wall_s", "mean"))
        .sort_values("mean_wall_s")
    )
    best_worker_by_case[case.case_id] = int(worker_summary.iloc[0]["requested_workers"])

    for chunk_size in case.chunk_candidates:
        all_rows.extend(
            run_profile(
                case,
                profile_name=f"chunk_sweep_{chunk_size // 1_000_000}mb",
                sharding_enabled=True,
                max_workers=best_worker_by_case[case.case_id],
                chunk_size_bp=chunk_size,
                repeats=case.repeats,
            )
        )

raw_results = pd.DataFrame(all_rows)
display(raw_results.head())
print("Rows collected:", len(raw_results))
print("Best worker by case:", best_worker_by_case)

In [ ]:
phase_cols = [
    "resolve_inputs_s",
    "load_fasta_s",
    "build_shards_s",
    "search_shards_s",
    "dedupe_s",
    "rank_s",
    "truncate_s",
]

summary = raw_results.groupby(
    [
        "case_id",
        "case_label",
        "profile_name",
        "sharding_enabled",
        "requested_workers",
        "chunk_size_bp",
    ],
    as_index=False,
).agg(
    mean_wall_s=("wall_s", "mean"),
    min_wall_s=("wall_s", "min"),
    max_wall_s=("wall_s", "max"),
    effective_workers=("effective_workers", "max"),
    shard_count=("shard_count", "max"),
    total_bp=("total_bp", "max"),
    raw_sites=("raw_sites", "mean"),
    deduped_sites=("deduped_sites", "mean"),
    ranked_sites=("ranked_sites", "mean"),
    retained_sites=("retained_sites", "mean"),
    **{phase: (phase, "mean") for phase in phase_cols},
)

baseline_map = summary.loc[summary["profile_name"] == "baseline_unsharded", ["case_id", "mean_wall_s"]].rename(
    columns={"mean_wall_s": "baseline_unsharded_s"}
)
serial_map = summary.loc[summary["profile_name"] == "sharded_serial", ["case_id", "mean_wall_s"]].rename(
    columns={"mean_wall_s": "sharded_serial_s"}
)
summary = summary.merge(baseline_map, on="case_id", how="left").merge(serial_map, on="case_id", how="left")
summary["speedup_vs_unsharded"] = summary["baseline_unsharded_s"] / summary["mean_wall_s"]
summary["speedup_vs_sharded_serial"] = summary["sharded_serial_s"] / summary["mean_wall_s"]
summary["parallel_efficiency"] = summary["speedup_vs_sharded_serial"] / summary["effective_workers"].clip(lower=1)

phase_rows: list[dict[str, Any]] = []
for row in summary.to_dict(orient="records"):
    total = float(row["mean_wall_s"])
    for phase in phase_cols:
        phase_time = float(row[phase])
        remaining = max(1e-9, total - phase_time)
        phase_rows.append(
            {
                "case_id": row["case_id"],
                "profile_name": row["profile_name"],
                "phase": phase,
                "phase_s": phase_time,
                "phase_share_pct": (phase_time / total) * 100.0,
                "max_speedup_if_phase_free": total / remaining,
            }
        )
phase_breakdown = pd.DataFrame(phase_rows)

worker_summary = summary[summary["profile_name"].str.startswith("worker_sweep_w")].copy()
chunk_summary = summary[summary["profile_name"].str.startswith("chunk_sweep_")].copy()
best_profiles = summary.sort_values(["case_id", "mean_wall_s"]).groupby("case_id", as_index=False).first()
best_profile_names = set(best_profiles["profile_name"])
best_phase_breakdown = phase_breakdown[phase_breakdown["profile_name"].isin(best_profile_names)].copy()

print("Best observed profiles")
display(
    best_profiles[
        [
            "case_id",
            "profile_name",
            "requested_workers",
            "effective_workers",
            "chunk_size_bp",
            "mean_wall_s",
            "speedup_vs_unsharded",
        ]
    ]
)
print("\nWorker sweep summary")
display(
    worker_summary[
        [
            "case_id",
            "requested_workers",
            "effective_workers",
            "mean_wall_s",
            "speedup_vs_unsharded",
            "speedup_vs_sharded_serial",
            "parallel_efficiency",
        ]
    ].sort_values(["case_id", "requested_workers"])
)
print("\nChunk sweep summary")
display(
    chunk_summary[
        [
            "case_id",
            "chunk_size_bp",
            "requested_workers",
            "effective_workers",
            "mean_wall_s",
            "speedup_vs_unsharded",
        ]
    ].sort_values(["case_id", "chunk_size_bp"])
)
print("\nTop bottlenecks for best observed profile per case")
display(
    best_phase_breakdown.sort_values(
        ["case_id", "max_speedup_if_phase_free"],
        ascending=[True, False],
    )
    .groupby("case_id")
    .head(5)
)

In [ ]:
if not worker_summary.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.lineplot(
        data=worker_summary.sort_values(["case_id", "requested_workers"]),
        x="requested_workers",
        y="mean_wall_s",
        hue="case_id",
        marker="o",
        ax=axes[0],
    )
    axes[0].set_title("Worker sweep wall time")
    axes[0].set_xlabel("Requested workers")
    axes[0].set_ylabel("Mean wall time (s)")

    sns.lineplot(
        data=worker_summary.sort_values(["case_id", "requested_workers"]),
        x="requested_workers",
        y="speedup_vs_sharded_serial",
        hue="case_id",
        marker="o",
        ax=axes[1],
        legend=False,
    )
    axes[1].set_title("Worker sweep speedup over sharded serial")
    axes[1].set_xlabel("Requested workers")
    axes[1].set_ylabel("Speedup")
    plt.tight_layout()
    plt.show()

if not best_phase_breakdown.empty:
    phase_plot = best_phase_breakdown.copy()
    phase_plot["phase"] = phase_plot["phase"].str.replace("_s", "", regex=False)
    plt.figure(figsize=(12, 5))
    sns.barplot(
        data=phase_plot.sort_values(["case_id", "phase_share_pct"], ascending=[True, False]),
        x="phase",
        y="phase_share_pct",
        hue="case_id",
    )
    plt.title("Phase share for best observed profile")
    plt.ylabel("Share of wall time (%)")
    plt.xlabel("Phase")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
raw_path = ARTIFACT_ROOT / "raw_results.csv"
summary_path = ARTIFACT_ROOT / "summary.csv"
phase_path = ARTIFACT_ROOT / "phase_breakdown.csv"
best_path = ARTIFACT_ROOT / "best_profiles.json"

raw_results.to_csv(raw_path, index=False)
summary.to_csv(summary_path, index=False)
phase_breakdown.to_csv(phase_path, index=False)
best_path.write_text(best_profiles.to_json(orient="records", indent=2), encoding="utf-8")

print(raw_path)
print(summary_path)
print(phase_path)
print(best_path)